# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)

In [ ]:
from pyspark.sql import functions as F

df = spark.read.option("header", "true").csv(
    "s3://streaming-mimic/sourceData/chartevents/"
)

df = df.withColumn(
    "charttime",
    F.to_timestamp("charttime")
)

min_time = df.select(F.min("charttime")).collect()[0][0]

# 30 seconds for chartevents batching
df = df.withColumn(
    "batch_id",
    F.floor(
        (
            F.unix_timestamp("charttime")
            - F.unix_timestamp(F.lit(min_time))
        ) / 1800
    )
)

df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("batch_id") \
    .csv("s3://streaming-mimic/data-streams/chartevents-streams/")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: e6f41248-c158-4519-93fd-eac24bee3c7e
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session e6f41248-c158-4519-93fd-eac24bee3c7e to get into ready status...
Session e6f41248-c158-4519-93fd-eac24bee3c7e has been created.


In [ ]:
df["batch_id"].max()

In [ ]:
from pyspark.sql import functions as F
import boto3

bucket = "streaming-mimic"
base_path = "data-streams/chartevents-streams/"

df = spark.read.option("header", "true").csv(
    "s3://streaming-mimic/sourceData/chartevents/"
)

df = df.withColumn("charttime", F.to_timestamp("charttime"))

min_time = df.agg(F.min("charttime")).first()[0]

# 30m for chartevents batching
df = df.withColumn(
    "batch_id",
    F.floor(
        (F.unix_timestamp("charttime") - F.unix_timestamp(F.lit(min_time))) / 1800
    )
)

batch_ids = [r[0] for r in df.select("batch_id").distinct().collect()]

s3 = boto3.client("s3")

for b in sorted(batch_ids):

    batch_df = df.filter(F.col("batch_id") == b).drop("batch_id")

    # write temporary folder
    tmp_path = f"s3://{bucket}/{base_path}tmp/batch_{b}/"

    batch_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(tmp_path)

    # rename part file to real csv file
    response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{base_path}tmp/batch_{b}/")

    for obj in response.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".csv"):
            copy_source = {"Bucket": bucket, "Key": key}

            final_key = f"{base_path}batch_{b}.csv"

            s3.copy_object(
                Bucket=bucket,
                CopySource=copy_source,
                Key=final_key
            )

    # delete temp folder
    for obj in response.get("Contents", []):
        s3.delete_object(Bucket=bucket, Key=obj["Key"])

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: b3c9581f-a5cb-4c5d-9130-2b4c21bb39b9
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session b3c9581f-a5cb-4c5d-9130-2b4c21bb39b9 to get into ready status...
Session b3c9581f-a5cb-4c5d-9130-2b4c21bb39b9 has been created.


In [ ]:
from pyspark.sql import functions as F
import boto3

bucket = "streaming-mimic"
base_path = "data-streams/outputevents-streams/"

df = spark.read.option("header", "true").csv(
    "s3://streaming-mimic/sourceData/outpurevents/"
)

df = df.withColumn("charttime", F.to_timestamp("charttime"))

min_time = df.agg(F.min("charttime")).first()[0]

# 12h for outputevents batching
df = df.withColumn(
    "batch_id",
    F.floor(
        (F.unix_timestamp("charttime") - F.unix_timestamp(F.lit(min_time))) / 43200
    )
)

batch_ids = [r[0] for r in df.select("batch_id").distinct().collect()]

s3 = boto3.client("s3")

for b in sorted(batch_ids):

    batch_df = df.filter(F.col("batch_id") == b).drop("batch_id")

    # tmp folder
    tmp_path = f"s3://{bucket}/{base_path}tmp/batch_{b}/"

    batch_df.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv(tmp_path)

    # get generated file
    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=f"{base_path}tmp/batch_{b}/"
    )

    for obj in response.get("Contents", []):
        key = obj["Key"]

        if key.endswith(".csv"):
            copy_source = {"Bucket": bucket, "Key": key}

            final_key = f"{base_path}batch_{b}.csv"

            s3.copy_object(
                Bucket=bucket,
                CopySource=copy_source,
                Key=final_key
            )

    # delete temp files
    for obj in response.get("Contents", []):
        s3.delete_object(Bucket=bucket, Key=obj["Key"])

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 7f6f3174-ffce-473e-bb27-26e2b6da31cf
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 7f6f3174-ffce-473e-bb27-26e2b6da31cf to get into ready status...
Session 7f6f3174-ffce-473e-bb27-26e2b6da31cf has been created.
{'ResponseMetadata': {'RequestId': 'KDJHGZJYN4H945E4', 'HostId': 'uZdUk3/nSEKRK43tycY+OBTOEaGnyeNeBXN+GaJxInJcxVEf9BWouWuSOoYfDuP4Eo1mus49H+gue5HFBK6rr82z0TeO4Yhx', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'uZdUk3/nSEKRK43tycY+OBTOEaGnyeNeBXN+GaJxInJcxVEf9BWouWuSOoYf